# Makemore Part 3：Activations & Gradients, BatchNorm

本 notebook 是 Part 3 的可运行前置代码。它复用 Part 2 的字符级 MLP 与数据划分，并为后续观察激活值、梯度、Batch Normalization 和参数初始化做好准备。

## 本节目标

- 构建训练 / 验证 / 测试集，并建立固定窗口字符语言模型。
- 检查激活值、梯度与参数更新的分布，识别饱和、梯度过大或过小等训练问题。
- 理解并手写 BatchNorm，再与 PyTorch 实现对照。

In [1]:
import random

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

torch.set_default_dtype(torch.float32)

In [2]:
# 读取与 Part 1、Part 2 相同的 names 数据集
words = open('names.txt', 'r', encoding='utf-8').read().splitlines()
print(f'{len(words)=}')
print(words[:8])

len(words)=32033
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
# 字符表：0 保留给起止标记 '.'
chars = sorted(set(''.join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)

print(stoi)
print(f'{vocab_size=}')

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
vocab_size=27


In [4]:
# 用连续 block_size 个字符预测下一个字符
block_size = 3

def build_dataset(words):
    X, Y = [], []
    for word in words:
        context = [0] * block_size
        for ch in word + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

# 与官方 notebook 一致：固定随机顺序后再切分，避免按字母排序带来分布偏差
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

print(f'{Xtr.shape=}, {Ytr.shape=}') # 80
print(f'{Xdev.shape=}, {Ydev.shape=}')# 10
print(f'{Xte.shape=}, {Yte.shape=}')# 10

Xtr.shape=torch.Size([182625, 3]), Ytr.shape=torch.Size([182625])
Xdev.shape=torch.Size([22655, 3]), Ydev.shape=torch.Size([22655])
Xte.shape=torch.Size([22866, 3]), Yte.shape=torch.Size([22866])


In [ ]:
# Part 3 的起点：MLP + 手写 BatchNorm 的参数与状态
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g)
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.zeros(vocab_size)

# BatchNorm 的可学习仿射参数；第一层偏置 b1 在 BN 后会被抵消，故不需要。
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
# 训练时更新；推理 / 采样时使用，避免依赖当前 mini-batch。
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

print(f'parameter count: {sum(p.nelement() for p in parameters):,}')
max_steps = 2000
for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), genetator = g)
    Vb, Yb = Xtr[ix], Ytr[ix]
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    hpreact = bngain * (hpreact - hpreact.mean(0, keepdim = True)) / hpreact.std(0, keepdim = True) + bnbias
    h = torch.tanh(hpreact)

parameter count: 12,097


In [6]:
import torch

# 假设输入一个 Batch 的图像 Patch 或 Token Embedding
# x 维度: [Batch, In_features]
x = torch.randn(32, 1024)  # Var(x) ≈ 1.0

# 错误做法：没有缩放
W_bad = torch.randn(1024, 2048)
y_bad = x @ W_bad
print(f"未初始化方差: {y_bad.var().item():.2f}")  # 约 1024 左右，严重爆炸！

# 正确做法：Kaiming Normal (针对 ReLU)
# std = sqrt(2 / in_features)
std = (2.0 / 1024) ** 0.5
W_good = torch.randn(1024, 2048) * std
y_good = x @ W_good
print(f"Kaiming 初始化后方差: {y_good.var().item():.2f}")  # 约 2.0，经过 ReLU 后刚好降回 1.0

未初始化方差: 1027.82
Kaiming 初始化后方差: 1.99
